# Notebook Dedicated to read Pre-filtered RATDS structure ROOT files and save Observables of Interest

In [2]:
import uproot
import numpy as np
import glob
import os

In [8]:
f = uproot.open(flist[0])
print(f.keys())

['T;1', 'pmt;1']


# Load Data

In [19]:
main_dir = '/home/joankl/data/solars/real_data/bisMSB/first_candidates/ratds/analysis*/ratds_data/*.root'
out_file_dir = '/home/joankl/data/solars/real_data/bisMSB/first_candidates/np_files/'
flist = glob.glob(main_dir)

# ======= Analysis Panel control =======

# ------- TTree entries to be used -------
read_var_list = ['evtid', 'sun_dir', 'hit_pmtid']  # Variables to read and use along the analysis
save_var_list = ['evtid', 'sun_dir'] # Variables to save
pmt_var_list = ['pmt_id', 'pmt_pos_xyz', 'pmt_type'] # PMT Info. to read

# ======================================

for fi_dx, file_i in enumerate(flist):
    print(f'In file {file_i}')

    # ------- Create empty lsit for each observable that is not part of the TTree -------
    multi_cos_alpha = []
    
    # ------- Load data and Select keys -------
    f_load = uproot.open(file_i)

    print(f'keys of loaded file {f_load.keys()}')
    
    #select the tree of event data and PMT info
    TTree_data_name = f_load.keys()[0]
    TTree_pmt_info_name = f_load.keys()[-1]
    
    event_data = f_load[TTree_data_name]
    pmt_data = f_load[TTree_pmt_info_name]

    observables = {}  # Dictioanry to save observables info. (data + PMT DB)
    
    # ------- import events Info. ------
    for var_i in read_var_list:
        #print(var_i)
        observables[var_i] = np.array(event_data[var_i])

    # ------- import PMT Info. -------
    try:
        for var_i in pmt_var_list:
            observables[var_i] = np.array(pmt_data[var_i])

    except uproot.KeyInFileError as e:
        print('Error in PMT Branch! Looking at other file')
        #if the Branch doesnt exist, then use the PMT info from other file that we know contais the PMT info Branch
        f_load = uproot.open(flist[0])
        pmt_data = f_load['pmt;1']
        for var_i in var_pmt_list:
            observables[var_i] = np.array(pmt_data[var_i])

    # ---------- Data Cuts ----------

    # PMT hit type selection (type = 1)
    pmt_type_condition = (observables['pmt_type'] == 1)
    pmt_id_valid = observables['pmt_id'][pmt_type_condition]

    hit_pmt_id_condition = np.in1d(observables['hit_pmtid'], pmt_id_valid)

    general_condition = hit_pmt_id_condition

    for var_i in read_var_list:
        observables[var_i] = observables[var_i][np.array(general_condition)]
        
    print(f'saving observables from list {read_var_list}')
    for var_i in read_var_list:
        np.save(out_file_dir + var_i + f'_{fi_dx}.npy', observables[var_i])

    # ---------- directional observable computation ----------
    
    N_samples = len(observables['evtid'])

    for sample_idx in range(N_samples):
        sun_dir = observables['sun_dir'][sample_idx]
        pmt_hit_id = observables['hit_pmtid'][sample_idx]
        pmt_hit_xyz = observables['pmt_pos_xyz'][pmt_hit_id]

        norm1 = np.linalg.norm(sun_dir)
        norm2 = np.linalg.norm(pmt_hit_xyz)
        
        sun_dir = sun_dir / norm1
        sun_dir = sun_dir.astype(np.float32)
        
        pmt_hit_xyz = pmt_hit_xyz / norm2
        pmt_hit_xyz = pmt_hit_xyz.astype(np.float32)
        
        dot_prod = np.dot(sun_dir, pmt_hit_xyz)
        cos_alpha = dot_prod
    
        multi_cos_alpha.append(cos_alpha)

    multi_cos_alpha = np.array(multi_cos_alpha)

    print('saving cos_alpha')
    np.save(out_file_dir +'cos_alpha' + f'_{fi_dx}.npy', multi_cos_alpha)

In file /home/joankl/data/solars/real_data/bisMSB/first_candidates/ratds/analysis20_bMR/ratds_data/analysis_ratds_sum_0.root
keys of loaded file ['T;1', 'pmt;1']
saving observables from list ['evtid', 'sun_dir', 'hit_pmtid']
saving cos_alpha
In file /home/joankl/data/solars/real_data/bisMSB/first_candidates/ratds/analysis20_bMR/ratds_data/analysis_ratds_sum_30.root
keys of loaded file ['T;1', 'pmt;1']
saving observables from list ['evtid', 'sun_dir', 'hit_pmtid']
saving cos_alpha
In file /home/joankl/data/solars/real_data/bisMSB/first_candidates/ratds/analysis20_bMR/ratds_data/analysis_ratds_sum_13.root
keys of loaded file ['T;1', 'pmt;1']
saving observables from list ['evtid', 'sun_dir', 'hit_pmtid']
saving cos_alpha
In file /home/joankl/data/solars/real_data/bisMSB/first_candidates/ratds/analysis20_bMR/ratds_data/analysis_ratds_sum_62.root
keys of loaded file ['T;1', 'pmt;1']
saving observables from list ['evtid', 'sun_dir', 'hit_pmtid']


/tmp/ipykernel_141107/1218938774.py:58: DeprecationWarning: `in1d` is deprecated. Use `np.isin` instead.
  hit_pmt_id_condition = np.in1d(observables['hit_pmtid'], pmt_id_valid)


saving cos_alpha
In file /home/joankl/data/solars/real_data/bisMSB/first_candidates/ratds/analysis20_bMR/ratds_data/analysis_ratds_sum_59.root
keys of loaded file ['T;1', 'pmt;1']
saving observables from list ['evtid', 'sun_dir', 'hit_pmtid']
saving cos_alpha
In file /home/joankl/data/solars/real_data/bisMSB/first_candidates/ratds/analysis20_bMR/ratds_data/analysis_ratds_sum_32.root
keys of loaded file ['T;1', 'pmt;1']
saving observables from list ['evtid', 'sun_dir', 'hit_pmtid']
saving cos_alpha
In file /home/joankl/data/solars/real_data/bisMSB/first_candidates/ratds/analysis20_bMR/ratds_data/analysis_ratds_sum_42.root
keys of loaded file ['T;1', 'pmt;1']
saving observables from list ['evtid', 'sun_dir', 'hit_pmtid']
saving cos_alpha
In file /home/joankl/data/solars/real_data/bisMSB/first_candidates/ratds/analysis20_bMR/ratds_data/analysis_ratds_sum_43.root
keys of loaded file ['T;1', 'pmt;1']
saving observables from list ['evtid', 'sun_dir', 'hit_pmtid']
saving cos_alpha
In file /ho